# **BioGen AI — Colab GPU Backend (v5)**

Runs the full protein-design pipeline on a Colab GPU and exposes it to your **BioGen AI web app** through a public ngrok URL:

| Stage | Tool | What it does |
|---|---|---|
| 1/3 | **RFdiffusion** | generates de novo backbones (binder / motif / unconditional) |
| 2/3 | **ProteinMPNN** | designs amino-acid sequences for the backbones |
| 3/3 | **AlphaFold2** | predicts & validates the folded structures |

**How to use**
1. `Runtime` → `Change runtime type` → **T4 GPU** → Save
2. Upload `biogen_colab_backend_v5.py` to this Colab session (folder icon on the left)
3. Run the cells below in order
4. Copy the printed **PUBLIC URL** and **API KEY** into the BioGen app → **Settings** → *Test Connection* → *Save & Connect*
5. In the app: load a target structure, click **Generate designs** — Colab runs backbone → sequence → structure automatically and the results appear in the app's **Results** page

The first run installs RFdiffusion + weights (~3-5 min); the app's health indicator turns ready automatically when done. Keep this tab open while designing. To stop the server: interrupt the last cell.

In [ ]:
#@title 1. Check GPU
!nvidia-smi -L
import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("\n⚠️  NO GPU! Go to Runtime → Change runtime type → select T4 GPU, then re-run.")

In [ ]:
#@title 2. Set your tokens {display-mode: "form"}
# The fields below come pre-filled - just run this cell as-is (or edit first).
# These are passed to the backend when cell 3 starts it.
import os

NGROK_TOKEN = "3ITzSATFlrtGFm9l2nzYcLyDatf_3qhwnX7Xf8vGLLEzcsAbz" #@param {type:"string"}
BIOGEN_API_KEY = "biogen-key-2026" #@param {type:"string"}

os.environ["NGROK_AUTHTOKEN"] = NGROK_TOKEN.strip()
os.environ["BIOGEN_API_KEY"] = BIOGEN_API_KEY.strip()

if NGROK_TOKEN.strip():
    print("ngrok token set.")
else:
    print("WARNING: No ngrok token - the backend will start WITHOUT a public URL.")
print("API key:", os.environ["BIOGEN_API_KEY"])


In [ ]:
#@title 3. Start the BioGen backend (keep this cell running)
# First run: installs RFdiffusion/ColabDesign/weights in the background (~3-5 min).
# The public URL + API key are printed below — paste them into the app Settings.
!python biogen_colab_backend_v5.py

**Troubleshooting**
- **"RFdiffusion is still installing"** in the app → wait for `[setup] RFdiffusion is ready` in the cell above, then generate again.
- **Connection failed in the app** → the Colab session probably restarted; re-run cells 1-3 and paste the new URL.
- **ngrok tunnel failed** → check your authtoken in cell 2 (free tokens: one tunnel at a time — stop other tunnels first).
- **Job failed** → the error appears in the app job status and in this cell's output (RFdiffusion log tail is included).
- Results (backbones, MPNN sequences, AlphaFold models, `.result.zip`) also stay in the Colab `outputs/` folder if you prefer downloading them manually.